In [33]:
from dotenv import load_dotenv
load_dotenv()

True

In [34]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START, END
from pydantic import BaseModel,Field
from typing import Literal
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import create_agent 
import requests
from langchain.tools import tool

In [35]:

# category: coding, google_search, weather 
class FlowState(BaseModel):
    question:str = Field(description="User asked question")
    category:Literal["coding","google_search","weather"] = Field(default="google_search")
    answer:str=Field(default="")

In [36]:
class QuestionCategory(BaseModel):
    category:Literal["coding","google_search","weather"] = Field(default="google_search")

In [43]:
llm=ChatGroq(model="openai/gpt-oss-20b")

search=GoogleSerperAPIWrapper()

google_agent = create_agent(
    model=llm,
    tools=[search.run],
    system_prompt="you are an agent and can search for any question on google"
)

# @tool
# def get_weather(city:str):
#     """It provide real time weather details for any city"""   
#     return f"The current temperature in {city} is 23.C"

@tool
def get_weather(city: str):
    """Provides the current real-time temperature for any city."""

    # 1. City name -> latitude & longitude
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    geo_response = requests.get(geo_url, params=geo_params)
    geo_data = geo_response.json()

    if "results" not in geo_data or not geo_data["results"]:
        return f"Could not find the city: {city}"

    location = geo_data["results"][0]

    latitude = location["latitude"]
    longitude = location["longitude"]
    city_name = location["name"]

    # 2. Get current weather
    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m",
        "temperature_unit": "celsius"
    }

    weather_response = requests.get(weather_url, params=weather_params)
    weather_data = weather_response.json()

    temperature = weather_data["current"]["temperature_2m"]

    return f"The current temperature in {city_name} is {temperature}°C"


weather_agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="you are an agent and can provide real time weather details"
)


def check_question_category(state:FlowState):
    st_llm=llm.with_structured_output(QuestionCategory)
    res=st_llm.invoke(f"I want to know the category of quesion, question is {state.question}.If you are not sure then just give 'google_search' as a category")
    state.category=res.category
    return state

In [45]:
def route(state:FlowState)->Literal["coding","google_search","weather"]:
    return state.category

In [46]:
def coding(state:FlowState):
    print("coding")
    res=llm.invoke(f"you are a coding expert:{state.question}")
    state.answer=res.content
    return state

def weather(state:FlowState):
    print("weather")
    res = weather_agent.invoke({"messages":[
        {"role":"user", "content":state.question}
    ]})
    state.answer = res["messages"][-1].content
    return state
  

def google_search(state:FlowState):
    print("gogole search ")
    res = google_agent.invoke({"messages":[
        {"role":"user", "content":state.question}
    ]})
    state.answer = res["messages"][-1].content
    return state
    

In [49]:
graph = StateGraph(FlowState)
graph.add_node("check_question_category",check_question_category)
graph.add_node("coding", coding)
graph.add_node("weather", weather)
graph.add_node("google_search",google_search)

graph.add_edge(START,"check_question_category")
graph.add_conditional_edges("check_question_category",route)
graph.add_edge("coding",END)
graph.add_edge("weather",END)
graph.add_edge("google_search",END)

graph=graph.compile()


graph.invoke({"question":"weather of ludhiana"})

weather


{'question': 'weather of ludhiana',
 'category': 'weather',
 'answer': '**Weather in Ludhiana (as of now):**  \n- Current temperature: **29.8\u202f°C**  \n\nLet me know if you’d like additional details (e.g., humidity, wind, forecast).'}